# 数据读取 （data from Unit03_1_5_select.mat）

## 原始数据的读取

In [95]:
import scipy.io
import numpy as np

# ==================== 1️⃣ 读取 .mat 文件 ====================
mat_data = scipy.io.loadmat(
    '/home/charles/HZU/Data_processed/multi-condition-transfer-learning/Unit03_1_5_select_3.mat'
)

data = mat_data['Unit03_1_5_select_3']

print("Original data shape:", data.shape)  # (N, 14)

# ==================== 2️⃣ 设置抽样参数 ====================
n = 10000        # ⭐你只需要改这里
seed = 42      # 可选：保证可复现

np.random.seed(seed)

num_samples = data.shape[0]
assert n <= num_samples, "n cannot be larger than total samples!"

# ==================== 3️⃣ 随机抽取 n 个样本（行） ====================
indices = np.random.choice(num_samples, size=n, replace=False)
data = data[indices, :]

print("Sampled data shape:", data.shape)

# ==================== 4️⃣ 查看抽样结果 ====================
print(data[:5])   # 前 5 行看看


Original data shape: (14000, 14)
Sampled data shape: (10000, 14)
[[ 5.43047428e+00  1.05059376e+01  2.31127650e-01 -1.40501881e+01
   2.50210800e+02  3.08711639e+02  5.54433655e+02  8.17137878e+02
   8.00413025e+02  8.91505981e+02  7.48925903e+02  7.47335205e+02
   7.47687561e+02  1.22015419e+01]
 [ 5.43321800e+00  1.05319557e+01  2.27354318e-01 -1.53432178e+01
   2.44723053e+02  3.09451508e+02  5.57394836e+02  8.27757263e+02
   8.06735474e+02  8.95841003e+02  7.50904175e+02  7.50025269e+02
   7.50035461e+02  1.33322163e+01]
 [ 6.86902332e+00  9.63033295e+00  2.11065754e-01 -1.29801950e+01
   2.62162231e+02  3.07413025e+02  5.59327209e+02  8.18008972e+02
   7.97732300e+02  8.90310486e+02  7.49348816e+02  7.47007019e+02
   7.47838074e+02  6.77845287e+00]
 [ 5.36784887e+00  1.06744299e+01  2.37428218e-01 -1.19700117e+01
   2.54997070e+02  3.07855316e+02  5.53981567e+02  8.24121216e+02
   8.05774719e+02  8.93992188e+02  7.48753174e+02  7.47304749e+02
   7.47581665e+02  9.15509033e+00]
 [ 

## 特征和标签的分离 & 特征归一化

In [96]:
import numpy as np

# 假设 'data' 是一个二维数组或矩阵
# 分离特征和标签

# 特征是除了最后一列的数据
X = data[:, :-1]  # 所有行，去除最后一列

# 标签是最后一列的数据
y = data[:, -1]  # 所有行，只取最后一列
y = y.reshape(-1, 1)

# # 查看特征和标签
# print("Features (X):")
# print(X[:5])  # 查看前5个特征样本
# print("Labels (y):")
# print(y[:5])  # 查看前5个标签

# 查看特征和标签的形状
print("Shape of Features (X):", X.shape)
print("Shape of Labels (y):", y.shape)

# Z-score 标准化
X_mean = X.mean(axis=0)
X_std  = X.std(axis=0) + 1e-8   # 防止除 0

X_norm = (X - X_mean) / X_std

print("Normalized X shape:", X_norm.shape)


Shape of Features (X): (10000, 13)
Shape of Labels (y): (10000, 1)
Normalized X shape: (10000, 13)


## 三集划分

In [97]:
import numpy as np
from sklearn.model_selection import train_test_split

# 假设 X 和 y 是已经分离好的特征和标签
# X: 特征数据，y: 标签数据

# 设置随机种子，确保结果可复现
random_seed = 42

# 控制三集的划分比例：例如 70% 训练集，15% 验证集，15% 测试集
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 确保划分比例之和为1
assert train_ratio + val_ratio + test_ratio == 1.0, "The sum of ratios must be 1."

# 第一次划分，将训练集和验证+测试集合并
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=random_seed)

# 第二次划分，将验证集和测试集分开
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_ratio / (val_ratio + test_ratio), random_state=random_seed)

# 打印各个数据集的形状
print("Shape of Training Set (X_train, y_train):", X_train.shape, y_train.shape)
print("Shape of Validation Set (X_val, y_val):", X_val.shape, y_val.shape)
print("Shape of Test Set (X_test, y_test):", X_test.shape, y_test.shape)


Shape of Training Set (X_train, y_train): (6999, 13) (6999, 1)
Shape of Validation Set (X_val, y_val): (1500, 13) (1500, 1)
Shape of Test Set (X_test, y_test): (1501, 13) (1501, 1)


# 表征学习

## CPC模型

In [98]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    """
        编码基础单元
    """
    def __init__(self, in_dim, out_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, x):
        """
        x: [k, D]
        return: [out_dim]
        """
        x = x.reshape(-1)      # [k*D]
        return self.net(x)
    
def encode_window(encoder, X_window):
    """
    X_window: [k+1, D]
    return:
        h_list: list of [out_dim]
        h_t:    [out_dim]
    """
    h_list = []
    for i in range(X_window.shape[0]):
        h = encoder(X_window[i:i+1])  # [1, D] -> [out_dim]
        h_list.append(h)

    h_t = h_list[-1]          # 时间 t 的表示
    h_past = h_list[:-1]      # 前 k 个
    return h_past, h_t


import torch
import torch.nn.functional as F

def causal_weights_last_vs_past_h(h_past, h_t, lam=1e-3, tau=1.0):
    """
    h_past : list[Tensor]，每个 [out_dim]
    h_t    : Tensor [out_dim]
    lam    : 岭回归正则
    tau    : softmax 温度

    return:
        scores  : Tensor [k]
        weights : Tensor [k]，和为 1
    """
    scores = []

    for h_i in h_past:
        # 标量 a_i，使 a_i * h_{t-i} 最接近 h_t
        denom = torch.dot(h_i, h_i) + lam
        a = torch.dot(h_i, h_t) / denom

        h_hat = a * h_i
        err = torch.norm(h_t - h_hat, p=2)

        # 因果分数：预测误差越小，因果越强
        scores.append(1.0 / (err + 1e-6))

    scores = torch.stack(scores)           # [k]
    weights = F.softmax(scores / tau, dim=0)

    return scores, weights


def fuse_h_past(h_past, weights):
    """
    h_past : list[Tensor]，每个形状 [out_dim]
    weights: Tensor，形状 [k]，和为 1
    return : Tensor，形状 [out_dim]
    """
    # 确保类型一致
    weights = weights.to(h_past[0].device)
    out = torch.zeros_like(h_past[0])
    for w, h in zip(weights, h_past):
        out = out + w * h
    return out

def process_window_with_causal_fusion(encoder, X_window, lam=1e-3, tau=1.0):
    """
    encoder : Encoder
    X_window: Tensor [k+1, D]，最后一行是 t 时刻
    lam     : 因果权重计算中的岭回归正则
    tau     : softmax 温度

    return:
        c_t : Tensor [out_dim]   # 因果加权融合后的特征
        h_t : Tensor [out_dim]   # 当前时刻的特征
    """

    # 1️⃣ 编码窗口内所有样本，得到 h_past 和 h_t
    h_past, h_t = encode_window(encoder, X_window)

    # 2️⃣ 用原始数据计算 past -> t 的因果权重
    scores, weights = causal_weights_last_vs_past_h(
        h_past,
        h_t,
        lam=1e-3,
        tau=tau
    )

    # 3️⃣ 根据因果权重融合 h_past
    c_t = fuse_h_past(h_past, weights)

    return c_t, h_t



In [101]:
import torch
import torch.nn.functional as F

def cpc_loss_single(c_t, h_pos, h_neg_list, tau=0.5):
    """
    c_t        : [out_dim]
    h_pos      : [out_dim]
    h_neg_list : list of [out_dim]
    """
    # cosine similarity（已归一化）
    pos_score = torch.dot(c_t, h_pos) / tau

    neg_scores = torch.stack([
        torch.dot(c_t, h_neg) / tau
        for h_neg in h_neg_list
    ])

    logits = torch.cat(
        [pos_score.unsqueeze(0), neg_scores],
        dim=0
    ).unsqueeze(0)  # [1, 1+Nneg]

    labels = torch.zeros(1, dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, labels)

import random

def train_step_one_window(
    encoder,
    Xw,                 # Tensor [Nw, T, D]
    idx,                # 当前窗口索引
    num_neg=10,
    lam=1e-3,
    tau=0.5
):
    """
    返回：loss（标量）
    """

    # ========== 1️⃣ 正样本（当前窗口） ==========
    X_window = Xw[idx]
    c_t, h_t = process_window_with_causal_fusion(
        encoder, X_window, lam=lam, tau=1.0
    )

    # L2 归一化（防止塌缩）
    c_t = F.normalize(c_t, dim=0)
    h_t = F.normalize(h_t, dim=0)

    # ========== 2️⃣ 负样本窗口索引 ==========
    all_idxs = list(range(len(Xw)))
    all_idxs.remove(idx)
    neg_idxs = random.sample(all_idxs, num_neg)

    h_neg_list = []
    for j in neg_idxs:
        _, h_neg = process_window_with_causal_fusion(
            encoder, Xw[j], lam=lam, tau=1.0
        )
        h_neg = F.normalize(h_neg, dim=0)
        h_neg_list.append(h_neg)

    # ========== 3️⃣ CPC loss ==========
    loss = cpc_loss_single(
        c_t, h_t, h_neg_list, tau=tau
    )

    return loss


## CPC表征学习

### 用于表征学习的数据处理

In [102]:
import numpy as np
from collections import Counter

def sliding_window_with_majority_label(X, y, window_size, stride=1):
    """
    X: (N, D)
    y: (N,) or (N, 1)
    window_size: 窗口长度
    stride: 滑动步长

    return:
        X_windows: (num_windows, window_size, D)
        y_windows: (num_windows,)
    """
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    N, D = X.shape

    X_windows = []
    y_windows = []

    for start in range(0, N - window_size + 1, stride):
        end = start + window_size

        x_win = X[start:end]               # (window_size, D)
        y_win = y[start:end]               # (window_size,)

        # 众数（出现次数最多的标签）
        label = Counter(y_win).most_common(1)[0][0]

        X_windows.append(x_win)
        y_windows.append(label)

    X_windows = np.stack(X_windows)        # (num_windows, window_size, D)
    y_windows = np.array(y_windows)        # (num_windows,)

    return X_windows, y_windows


window_size = 50
stride = 25

X_train_win, y_train_win = sliding_window_with_majority_label(
    X_train,
    y_train,
    window_size=window_size,
    stride=stride
)
y_train_win = y_train_win.reshape(-1, 1)

print(X_train_win.shape)   # (num_windows, 20, D)
print(y_train_win.shape)   # (num_windows,)


(278, 50, 13)
(278, 1)


### 表征学习训练

In [103]:
encoder = Encoder(in_dim=13, out_dim=64, hidden_dim=128)

device = "cuda" if torch.cuda.is_available() else "cpu"
encoder = encoder.to(device)

Xw = torch.tensor(X_train_win, dtype=torch.float32).to(device)

optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

num_epochs = 20
num_neg = 5

# ===== 固定一个 probe 窗口，用来观测权重变化 =====
probe_idx = 0
X_probe = Xw[probe_idx]

# （可选）保存权重历史
weight_history = []

for epoch in range(num_epochs):
    total_loss = 0.0
    encoder.train()

    for idx in range(len(Xw)):
        optimizer.zero_grad()

        loss = train_step_one_window(
            encoder,
            Xw,
            idx,
            num_neg=num_neg,
            lam=1e-3,
            tau=0.5
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(Xw)

    # ====== 报文：检查因果权重 ======
    encoder.eval()
    with torch.no_grad():
        h_past, h_t = encode_window(encoder, X_probe)
        scores, weights = causal_weights_last_vs_past_h(
            h_past, h_t, lam=1e-3, tau=1.0
        )

        weight_history.append(weights.cpu().numpy())

        print(
            f"[Epoch {epoch+1:03d}] "
            f"CPC Loss = {avg_loss:.4f} | "
            f"w_max = {weights.max().item():.4f}, "
            f"w_min = {weights.min().item():.4f}, "
            f"w_std = {weights.std().item():.4f}, "
            f"argmax = {weights.argmax().item()}"
        )


[Epoch 001] CPC Loss = 1.7917 | w_max = 0.0384, w_min = 0.0158, w_std = 0.0049, argmax = 21
[Epoch 002] CPC Loss = 1.7917 | w_max = 0.0432, w_min = 0.0142, w_std = 0.0064, argmax = 28


KeyboardInterrupt: 

# 图结构搭建

In [21]:
import numpy as np
import torch
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

# =========================================================
# 输入：
#   X : numpy.ndarray or torch.Tensor
#       shape = [num_samples, num_features]
# 输出：
#   adj : torch.Tensor
#       shape = [num_features, num_features]
# =========================================================

def build_feature_graph_from_X(X, threshold=0.8, device="cpu"):
    """
    Automatically build a feature (column-wise) graph from X.

    Parameters
    ----------
    X : np.ndarray or torch.Tensor
        Shape [num_samples, num_features]
    threshold : float
        Cosine similarity threshold for edge creation
    device : str or torch.device
        cpu / cuda

    Returns
    -------
    adj : torch.Tensor
        Adjacency matrix of shape [num_features, num_features]
    """

    # ---------- 1️⃣ 统一成 numpy ----------
    if isinstance(X, torch.Tensor):
        X_np = X.detach().cpu().numpy()
    else:
        X_np = X

    # ---------- 2️⃣ 自动读取形状 ----------
    num_samples, num_features = X_np.shape
    print(f"[INFO] X shape: samples={num_samples}, features={num_features}")

    # ---------- 3️⃣ 特征（列）作为节点 ----------
    # 每一列是一个节点向量（跨样本）
    X_feature = X_np.T                         # [F, M]

    # ---------- 4️⃣ 特征间相似度 ----------
    sim_matrix = cosine_similarity(X_feature) # [F, F]

    # ---------- 5️⃣ 构建 NetworkX 图 ----------
    G = nx.Graph()
    G.add_nodes_from(range(num_features))

    for i in range(num_features):
        for j in range(i + 1, num_features):
            if sim_matrix[i, j] >= threshold:
                G.add_edge(i, j, weight=sim_matrix[i, j])

    print(f"[INFO] Graph built: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")

    # ---------- 6️⃣ Graph → 邻接矩阵 ----------
    adj_np = nx.to_numpy_array(G, weight="weight")  # [F, F]

    # ---------- 7️⃣ 转成 torch.Tensor ----------
    adj = torch.tensor(adj_np, dtype=torch.float32, device=device)

    # ---------- 8️⃣ 简单健壮性检查 ----------
    isolated = (adj.sum(dim=1) == 0).sum().item()
    if isolated > 0:
        print(f"[WARN] {isolated} isolated feature nodes detected "
              f"(consider lowering threshold or using KNN graph)")

    print(f"[INFO] adj shape: {adj.shape}")
    return adj


device = "cuda" if torch.cuda.is_available() else "cpu"

adj = build_feature_graph_from_X(X, threshold=0.8, device=device)

print(adj)


# 模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ======================================================
# 0️⃣ 固定随机种子
# ======================================================
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================================================
# 1️⃣ 工具函数：统一转成 torch.Tensor
# ======================================================
def to_tensor(x, device):
    if isinstance(x, np.ndarray):
        return torch.tensor(x, dtype=torch.float32, device=device)
    elif isinstance(x, torch.Tensor):
        return x.to(device)
    else:
        raise TypeError(f"Unsupported type: {type(x)}")

# ======================================================
# 2️⃣ GCN Layer（adj = F × F）
# ======================================================
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, adj):
        """
        x   : [F, in_dim]
        adj : [F, F]
        """
        F_dim = adj.shape[0]

        I = torch.eye(F_dim, device=adj.device)
        A_hat = adj + I

        deg = A_hat.sum(dim=1)
        deg_inv_sqrt = torch.pow(deg, -0.5)
        deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0
        D_inv_sqrt = torch.diag(deg_inv_sqrt)

        A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt
        out = A_norm @ x
        out = self.linear(out)
        return out

# ======================================================
# 3️⃣ 样本级 GCN 回归模型
# ======================================================
class SampleLevelGCNRegressor(nn.Module):
    def __init__(self, hidden_dim=64, out_dim=1):
        super().__init__()

        self.gcn1 = GCNLayer(1, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, hidden_dim)

        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, X_batch, adj):
        """
        X_batch : [B, F]
        adj     : [F, F]
        """
        B, F_dim = X_batch.shape
        outputs = []

        for b in range(B):
            x = X_batch[b].unsqueeze(1)      # [F,1]

            h = F.relu(self.gcn1(x, adj))    # [F,d]
            h = F.relu(self.gcn2(h, adj))    # [F,d]

            g = h.mean(dim=0)                # [d]
            y_hat = self.regressor(g)        # [out_dim]
            outputs.append(y_hat)

        return torch.stack(outputs, dim=0)   # [B,out_dim]

# ======================================================
# 4️⃣ 训练 / 验证函数
# ======================================================
def train_epoch(model, optimizer, criterion, X_data, y_data, adj, batch_size):
    model.train()
    num_samples = X_data.shape[0]
    perm = torch.randperm(num_samples, device=X_data.device)

    total_loss = 0.0

    for start in range(0, num_samples, batch_size):
        idx = perm[start:start + batch_size]
        Xb = X_data[idx]
        yb = y_data[idx]

        y_hat = model(Xb, adj)
        loss = criterion(y_hat, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * Xb.shape[0]

    return total_loss / num_samples


@torch.no_grad()
def eval_epoch(model, criterion, X_data, y_data, adj, batch_size):
    model.eval()
    num_samples = X_data.shape[0]
    total_loss = 0.0

    for start in range(0, num_samples, batch_size):
        Xb = X_data[start:start + batch_size]
        yb = y_data[start:start + batch_size]

        y_hat = model(Xb, adj)
        loss = criterion(y_hat, yb)

        total_loss += loss.item() * Xb.shape[0]

    return total_loss / num_samples

# ======================================================
# 6️⃣ 数据转 Tensor（关键！）
# ======================================================
X_train = to_tensor(X_train, device)
X_val   = to_tensor(X_val, device)
X_test  = to_tensor(X_test, device)

y_train = to_tensor(y_train, device)
y_val   = to_tensor(y_val, device)
y_test  = to_tensor(y_test, device)

adj = to_tensor(adj, device)

# 防呆检查
assert X_train.shape[1] == adj.shape[0], \
    f"Feature mismatch: X has {X_train.shape[1]}, adj is {adj.shape}"

# ======================================================
# 7️⃣ 训练配置
# ======================================================
model = SampleLevelGCNRegressor(hidden_dim=64, out_dim=1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

batch_size = 32
epochs = 50



In [ ]:
# ======================================================
# 8️⃣ 正式训练
# ======================================================
best_val = float("inf")
best_state = None

for epoch in range(1, epochs + 1):
    train_loss = train_epoch(
        model, optimizer, criterion,
        X_train, y_train, adj, batch_size
    )
    val_loss = eval_epoch(
        model, criterion,
        X_val, y_val, adj, batch_size
    )

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if epoch == 1 or epoch % 5 == 0:
        print(f"[Epoch {epoch:03d}] "
              f"Train MSE: {train_loss:.4f} | Val MSE: {val_loss:.4f}")

# ======================================================
# 9️⃣ 测试集评估
# ======================================================
model.load_state_dict(best_state)
model.to(device)

test_loss = eval_epoch(
    model, criterion,
    X_test, y_test, adj, batch_size
)

print(f"\n✅ Test MSE: {test_loss:.4f}")


In [23]:
save_path = "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/multi_condition_transfer_learning/transfer_learning_v1/result/model_save/best_gcn_model.pt"
torch.save(best_state, save_path)
print(f"Model saved to {save_path}")


# 测试

In [28]:
from sklearn.metrics import r2_score
import numpy as np
import torch

@torch.no_grad()
def evaluate_r2(model, X_data, y_data, adj, batch_size):
    model.eval()

    y_true_list = []
    y_pred_list = []

    num_samples = X_data.shape[0]

    for start in range(0, num_samples, batch_size):
        Xb = X_data[start:start + batch_size]
        yb = y_data[start:start + batch_size]

        y_hat = model(Xb, adj)  # [B, 1]

        # 🔑 关键：全部拉平成一维向量
        y_true_list.append(yb.view(-1).cpu().numpy())
        y_pred_list.append(y_hat.view(-1).cpu().numpy())

    # 一维拼接（不会受 batch 大小影响）
    y_true = np.concatenate(y_true_list, axis=0)
    y_pred = np.concatenate(y_pred_list, axis=0)

    r2 = r2_score(y_true, y_pred)
    return r2


r2 = evaluate_r2(
    model,
    X_test,
    y_test,
    adj,
    batch_size=batch_size
)

print(f"✅ Test R²: {r2:.4f}")
